# 0. Chains Overview — LCEL, the `|` Operator, and Runnables

Before the four chain shapes, let's understand **what a chain actually is**, the **Runnable interface**
that makes chains possible, and the **`|` (pipe) operator** that builds them.

---

## 1. Simple Definition

> **Kid version:** Think of a **toy marble run**. You drop a marble in the top, it rolls through one
> piece, drops into the next piece, then the next, and pops out the bottom. A **chain** is a marble run
> for data: your input rolls through step after step and comes out transformed.

**Professional definition:** A *chain* is a composition of components (prompts, models, parsers,
functions) where each component's output is passed as the next component's input. In modern LangChain,
chains are built with **LCEL** by connecting **Runnables** using the `|` operator.

```python
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

prompt = ChatPromptTemplate.from_template("Tell me a joke about {topic}")
chain = prompt | model | StrOutputParser()      # a 3-step chain
chain.invoke({"topic": "cats"})                  # "Why did the cat..."
```

---

## 2. Why Do Chains Exist?

**The problem:** Real tasks need **more than one step**. "Summarize this document, then translate the
summary, then extract keywords" is three model calls plus glue. Doing that by hand means manually
passing outputs around, handling batching, streaming, and async for each piece.

### Before chains (manual wiring)

```python
prompt_value = prompt.format_messages(topic="cats")   # 1) format
ai_message = model.invoke(prompt_value)                # 2) call model
text = ai_message.content                              # 3) unwrap
# ...now repeat for the next step, passing `text` along by hand 😩
```

### After chains (LCEL)

```python
chain = prompt | model | StrOutputParser()
chain.invoke({"topic": "cats"})     # all three steps, one call
```

Chains give you **composition** (snap steps together), plus **streaming, batching, async, and
retries** for free on the whole pipeline — because every piece shares one interface.

**Where you'll use it:** literally everywhere in LangChain — RAG, agents, extraction, summarization,
routing. Chains are the backbone.

---

## 3. Real-Life Analogy

**A factory assembly line** 🏭. Raw materials enter at one end. Station 1 shapes the part, passes it to
Station 2 which paints it, then Station 3 packages it. Each station does one job and hands its result
to the next. A chain is that line; each Runnable is a station.

Other analogies: a **relay race** (baton = data passed runner to runner), **Unix pipes**
(`cat file | grep x | sort`), a **kitchen line** (prep → cook → plate).

---

## 4. The Foundation: the Runnable Interface

Every LangChain building block — prompts, models, parsers, even your own functions and whole chains —
is a **Runnable**. They all share the same methods:

```
                       Runnable  (the shared interface)
                            │
   ┌──────────┬─────────────┼──────────────┬───────────────┐
   ▼          ▼             ▼              ▼               ▼
 Prompt      Model        Parser      RunnableLambda    a whole Chain
Templates  (ChatModel)  (OutputParser) (your function)  (RunnableSequence)
```

The core methods every Runnable has:

| Method | What it does |
|--------|--------------|
| `.invoke(input)` | Run on **one** input, return the output. |
| `.batch([...])` | Run on **many** inputs (parallelized). |
| `.stream(input)` | Run and **yield** output chunks as they're produced. |
| `.ainvoke` / `.abatch` / `.astream` | Async versions of the above. |

**This is the key idea:** because *everything* speaks this same interface, you can plug any Runnable
into any other. A chain is just a Runnable built from Runnables — so chains nest inside chains.

---

## 5. Internal Working — how `|` builds a chain

```
  prompt | model | parser
        │
        ▼
  The `|` operator calls prompt.__or__(model) → RunnableSequence(prompt, model)
        │
        ▼
  ... | parser → RunnableSequence(prompt, model, parser)
        │
        ▼
  A single RunnableSequence Runnable with 3 steps
        │
        ▼
  .invoke({"topic": "cats"}):
     step output → next step input, in order:
        {"topic":"cats"} → prompt → PromptValue → model → AIMessage → parser → "..."
```

So `|` doesn't run anything immediately — it **composes** a new Runnable. Execution happens when you
call `.invoke()` / `.stream()` / `.batch()`. The output type of each step must match (roughly) the
input type the next step expects — that's the one rule to keep in mind.

```python
type(prompt | model | parser)   # <class 'langchain_core.runnables.base.RunnableSequence'>
```

---

## 6. The building blocks you'll compose

### The `|` (pipe) operator

**Definition:** Connects two Runnables so the left's output feeds the right's input.

**Why it exists:** Readable, declarative composition (reads left-to-right like a pipeline).

**When developers use it:** To build essentially every chain.

```python
chain = prompt | model | StrOutputParser()
```

---

### RunnableSequence (the `"sequential"` engine)

**Definition:** The Runnable that runs steps **in order**, piping each output to the next. `|` creates
it for you.

**Why it exists:** It's the backbone of simple and sequential chains.

```python
from langchain_core.runnables import RunnableSequence
chain = RunnableSequence(prompt, model, parser)   # same as prompt | model | parser
```

---

### RunnableParallel (the `"parallel"` engine)

**Definition:** Runs multiple Runnables on the **same input at once**, returning a dict of results.

**Why it exists:** Fan-out work that doesn't depend on each other.

```python
from langchain_core.runnables import RunnableParallel
RunnableParallel(joke=joke_chain, poem=poem_chain)
```

---

### RunnableBranch (the `"conditional"` engine)

**Definition:** Picks **one** of several branches based on a condition (like if/elif/else).

**Why it exists:** Route different inputs to different sub-chains.

```python
from langchain_core.runnables import RunnableBranch
RunnableBranch((is_math, math_chain), (is_code, code_chain), default_chain)
```

---

### `RunnableLambda & RunnablePassthrough` (the glue)

**Definition:** `RunnableLambda` wraps any Python function as a Runnable; `RunnablePassthrough` passes
input through unchanged (often to keep original data alongside new results).

**Why they exist:** Custom logic and data-shaping between steps.

```python
from langchain_core.runnables import RunnableLambda, RunnablePassthrough
```

---

## 7. The four shapes at a glance

```
  SIMPLE          prompt ─► model ─► parser
                          one straight pipeline

  SEQUENTIAL      chainA ─► chainB ─► chainC
                          output of each stage feeds the next (multi-stage)

  PARALLEL               ┌─► summary ─┐
                  input ─┤            ├─► {summary, keywords, sentiment}
                         ├─► keywords ┤
                         └─► sentiment┘   branches run together, merged into a dict

  CONDITIONAL     input ─► router ─► if math   → math_chain
                                     elif code → code_chain
                                     else      → general_chain
```

All four are **just Runnables**, so you can nest them: a parallel step inside a sequential chain inside
a conditional branch, etc.